In [1]:
import polars as pl
import polars_st as st
from polars_st import geom, geometry
from sklearn.cluster import DBSCAN

In [5]:
data = pl.scan_parquet("../data/processed/firms_noaa20.parquet")

In [6]:
data.head().collect()

latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,acquired_at_utc
f64,f64,f64,f64,f64,str,i64,str,str,str,str,f64,f64,str,"datetime[μs, UTC]"
-8.79831,140.69357,331.79,0.44,0.46,"""2026-08-24""",349,"""N20""","""VIIRS""","""n""","""2.0NRT""",295.91,3.73,"""D""",2026-08-24 03:49:00 UTC
-8.71738,140.74724,330.69,0.43,0.46,"""2026-08-24""",349,"""N20""","""VIIRS""","""n""","""2.0NRT""",290.74,1.54,"""D""",2026-08-24 03:49:00 UTC
-8.34927,138.0038,336.22,0.33,0.55,"""2026-08-24""",349,"""N20""","""VIIRS""","""n""","""2.0NRT""",302.29,5.4,"""D""",2026-08-24 03:49:00 UTC
-8.3187,137.76657,332.43,0.34,0.56,"""2026-08-24""",349,"""N20""","""VIIRS""","""n""","""2.0NRT""",299.93,2.74,"""D""",2026-08-24 03:49:00 UTC
-8.30191,137.89424,338.57,0.33,0.55,"""2026-08-24""",349,"""N20""","""VIIRS""","""n""","""2.0NRT""",301.05,4.29,"""D""",2026-08-24 03:49:00 UTC


In [16]:
data = data.with_columns(
    geometry=st.point(
        pl.concat_arr("longitude", "latitude"),
        srid=4326
    )
)

prov = st.read_file("../data/boundaries/geoBoundaries-IDN-ADM1-provinces.geojson")
prov = prov.select(
    pl.col("shapeName").alias("province"),
    "geometry"
)

/tmp/ipykernel_233868/72675481.py:8: FutureWarning: from_arrow(<ArrowStreamExportable>) will return a Series instead of a DataFrame in 2.0. To avoid this warning, pass the ArrowStreamExportable to either `pl.DataFrame` or `pl.Series` instead based on your desired output type.
  prov = st.read_file("../data/boundaries/geoBoundaries-IDN-ADM1-provinces.geojson")


In [17]:
print(prov.schema)
print(prov.head())

Schema({'province': String, 'geometry': Binary})
shape: (5, 2)
┌────────────────────┬─────────────────────────────────┐
│ province           ┆ geometry                        │
│ ---                ┆ ---                             │
│ str                ┆ binary                          │
╞════════════════════╪═════════════════════════════════╡
│ Bali               ┆ b"\x01\x06\x00\x00\x20\xe6\x10… │
│ West Nusa Tenggara ┆ b"\x01\x06\x00\x00\x20\xe6\x10… │
│ Banten             ┆ b"\x01\x06\x00\x00\x20\xe6\x10… │
│ Central Java       ┆ b"\x01\x06\x00\x00\x20\xe6\x10… │
│ West Java          ┆ b"\x01\x03\x00\x00\x20\xe6\x10… │
└────────────────────┴─────────────────────────────────┘


In [27]:
joined = (data.collect()
          .st.sjoin(
                prov,
                how="left",
                predicate="contains"
            )
          .drop("geometry_right")
          )

/tmp/ipykernel_233868/2806564407.py:2: DeprecationWarning: the default behavior of `how='horizontal'` for `concat` is deprecated and will require equal heights in the next breaking release. Use `how='horizontal_extend'` to keep the current behavior.
(Deprecated in version 1.42.1)
  .st.sjoin(


In [28]:
print(joined.schema)
print(joined.head())

Schema({'latitude': Float64, 'longitude': Float64, 'bright_ti4': Float64, 'scan': Float64, 'track': Float64, 'acq_date': String, 'acq_time': Int64, 'satellite': String, 'instrument': String, 'confidence': String, 'version': String, 'bright_ti5': Float64, 'frp': Float64, 'daynight': String, 'acquired_at_utc': Datetime(time_unit='us', time_zone='UTC'), 'geometry': Binary, 'province': String})
shape: (5, 17)
┌──────────┬───────────┬────────────┬──────┬───┬──────────┬──────────────┬──────────────┬──────────┐
│ latitude ┆ longitude ┆ bright_ti4 ┆ scan ┆ … ┆ daynight ┆ acquired_at_ ┆ geometry     ┆ province │
│ ---      ┆ ---       ┆ ---        ┆ ---  ┆   ┆ ---      ┆ utc          ┆ ---          ┆ ---      │
│ f64      ┆ f64       ┆ f64        ┆ f64  ┆   ┆ str      ┆ ---          ┆ binary       ┆ str      │
│          ┆           ┆            ┆      ┆   ┆          ┆ datetime[μs, ┆              ┆          │
│          ┆           ┆            ┆      ┆   ┆          ┆ UTC]         ┆            

In [31]:
indo_only = pl.read_parquet("../data/processed/firms_noaa20_indonesia.parquet")

In [33]:
indo_only.head()

latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,acquired_at_utc,geometry,province,kabupaten_kota
f64,f64,f64,f64,f64,str,i64,str,str,str,str,f64,f64,str,"datetime[μs, UTC]",binary,str,str
-8.79831,140.69357,331.79,0.44,0.46,"""2026-08-24""",349,"""N20""","""VIIRS""","""n""","""2.0NRT""",295.91,3.73,"""D""",2026-08-24 03:49:00 UTC,"b""\x01\x01\x00\x00\x20\xe6\x10\x00\x00\x93o\xb6\xb91\x96a@\xb8#\x9c\x16\xbc\x98!\xc0""","""Papua""","""Merauke"""
-8.71738,140.74724,330.69,0.43,0.46,"""2026-08-24""",349,"""N20""","""VIIRS""","""n""","""2.0NRT""",290.74,1.54,"""D""",2026-08-24 03:49:00 UTC,"b""\x01\x01\x00\x00\x20\xe6\x10\x00\x00kH\xdcc\xe9\x97a@\xe5\x9bmnLo!\xc0""","""Papua""","""Merauke"""
-8.34927,138.0038,336.22,0.33,0.55,"""2026-08-24""",349,"""N20""","""VIIRS""","""n""","""2.0NRT""",302.29,5.4,"""D""",2026-08-24 03:49:00 UTC,"b""\x01\x01\x00\x00\x20\xe6\x10\x00\x002w-!\x1f@a@\xa6\xf2v\x84\xd3\xb2\x20\xc0""","""Papua""","""Merauke"""
-8.3187,137.76657,332.43,0.34,0.56,"""2026-08-24""",349,"""N20""","""VIIRS""","""n""","""2.0NRT""",299.93,2.74,"""D""",2026-08-24 03:49:00 UTC,"b""\x01\x01\x00\x00\x20\xe6\x10\x00\x00\x08\x03\xcf\xbd\x878a@lxz\xa5,\xa3\x20\xc0""","""Papua""","""Merauke"""
-8.30191,137.89424,338.57,0.33,0.55,"""2026-08-24""",349,"""N20""","""VIIRS""","""n""","""2.0NRT""",301.05,4.29,"""D""",2026-08-24 03:49:00 UTC,"b""\x01\x01\x00\x00\x20\xe6\x10\x00\x00\xcdX4\x9d\x9d<a@\xb4\xab\x90\xf2\x93\x9a\x20\xc0""","""Papua""","""Merauke"""


In [3]:
daily_kabkot = pl.read_parquet("../data/analytics/daily_kabupaten_kota.parquet")
daily_prov = pl.read_parquet("../data/analytics/daily_province.parquet")
firms30d = pl.read_parquet("../data/analytics/firms_30d.parquet")

In [43]:
print(daily_kabkot.shape)
display(daily_kabkot.head())
print(daily_prov.shape)
display(daily_prov.head())
print(firms30d.shape)
display(firms30d.head())

(6947, 7)


acq_date,province,kabupaten_kota,hotspot_count,high_confidence_count,total_frp,max_frp
str,str,str,u32,u32,f64,f64
"""2026-07-26""","""Papua""","""Merauke""",243,1,2161.03,132.06
"""2026-07-26""","""West Kalimantan""","""Sanggau""",138,0,2257.02,198.95
"""2026-07-26""","""West Kalimantan""","""Ketapang""",63,0,636.93,181.99
"""2026-07-26""","""West Kalimantan""","""Kayong Utara""",62,1,317.4,18.95
"""2026-07-26""","""Papua""","""Mappi""",59,0,465.21,30.18


(939, 6)


acq_date,province,hotspot_count,high_confidence_count,total_frp,max_frp
str,str,u32,u32,f64,f64
"""2026-07-26""","""West Kalimantan""",441,2,5424.03,198.95
"""2026-07-26""","""Papua""",354,1,2989.11,132.06
"""2026-07-26""","""Central Kalimantan""",144,7,1096.95,76.31
"""2026-07-26""","""East Java""",91,1,434.56,34.02
"""2026-07-26""","""Central Java""",68,0,237.98,29.55


(98245, 18)


latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,acquired_at_utc,geometry,province,kabupaten_kota
f64,f64,f64,f64,f64,str,i64,str,str,str,str,f64,f64,str,"datetime[μs, UTC]",binary,str,str
-8.81859,140.97943,328.55,0.42,0.45,"""2026-07-26""",434,"""N20""","""VIIRS""","""n""","""2.0NRT""",294.56,2.43,"""D""",2026-07-26 04:34:00 UTC,"b""\x01\x01\x00\x00\x20\xe6\x10\x00\x00\x15W\x95}W\x9fa@P\xaa}:\x1e\xa3!\xc0""","""Papua""","""Merauke"""
-8.81766,140.95972,341.96,0.42,0.45,"""2026-07-26""",434,"""N20""","""VIIRS""","""n""","""2.0NRT""",296.72,4.42,"""D""",2026-07-26 04:34:00 UTC,"b""\x01\x01\x00\x00\x20\xe6\x10\x00\x00&\xaa\xb7\x06\xb6\x9ea@\xa6~\xdeT\xa4\xa2!\xc0""","""Papua""","""Merauke"""
-8.81443,140.97874,328.47,0.42,0.45,"""2026-07-26""",434,"""N20""","""VIIRS""","""n""","""2.0NRT""",294.4,2.72,"""D""",2026-07-26 04:34:00 UTC,"b""\x01\x01\x00\x00\x20\xe6\x10\x00\x00/i\x8c\xd6Q\x9fa@7\xc3\x0d\xf8\xfc\xa0!\xc0""","""Papua""","""Merauke"""
-8.80832,140.98946,329.22,0.42,0.45,"""2026-07-26""",434,"""N20""","""VIIRS""","""n""","""2.0NRT""",294.32,5.82,"""D""",2026-07-26 04:34:00 UTC,"b""\x01\x01\x00\x00\x20\xe6\x10\x00\x00h\x96\x04\xa8\xa9\x9fa@\xcbgy\x1e\xdc\x9d!\xc0""","""Papua""","""Merauke"""
-8.70118,140.58792,339.31,0.4,0.44,"""2026-07-26""",434,"""N20""","""VIIRS""","""n""","""2.0NRT""",293.06,3.86,"""D""",2026-07-26 04:34:00 UTC,"b""\x01\x01\x00\x00\x20\xe6\x10\x00\x00B\x95\x9a=\xd0\x92a@\xf47\xa1\x10\x01g!\xc0""","""Papua""","""Merauke"""


In [6]:
firms30d.filter(
    pl.col("acq_date") == "2026-08-23",
    pl.col("province") == "East Kalimantan"
).height

306

In [48]:
(
    daily_kabkot
    .filter(
        pl.col("acq_date") == "2026-08-23"
    )
    .sort(
        "hotspot_count",
        descending=True,
    )
    .head(10)
)

acq_date,province,kabupaten_kota,hotspot_count,high_confidence_count,total_frp,max_frp
str,str,str,u32,u32,f64,f64
"""2026-08-23""","""Papua""","""Merauke""",560,3,3713.1,75.77
"""2026-08-23""","""South Sumatra""","""Ogan Komering Ilir""",279,43,5412.3,126.32
"""2026-08-23""","""West Kalimantan""","""Ketapang""",227,4,1442.2,198.53
"""2026-08-23""","""Papua""","""Mappi""",180,0,1938.99,149.06
"""2026-08-23""","""East Kalimantan""","""Berau""",132,0,1716.87,163.78
"""2026-08-23""","""South Sumatra""","""Musi Banyuasin""",124,8,1064.05,67.14
"""2026-08-23""","""Central Kalimantan""","""Kotawaringin Timur""",111,1,286.41,15.0
"""2026-08-23""","""Maluku""","""Seram Bagian Timur""",105,0,181.56,5.71
"""2026-08-23""","""Riau""","""Indragiri Hilir""",104,9,700.97,54.58


In [53]:
clusterz = pl.read_parquet("../data/analytics/hotspot_clusters.parquet")

In [57]:
clusterz.select(
    "detection_count",
    "active_days",
    "track_detection_count",
).describe()

(
    clusterz
    .sort("detection_count", descending=True)
    .select(
        "acq_date",
        "cluster_id",
        "province",
        "kabupaten_kota",
        "detection_count",
        "centroid_latitude",
        "centroid_longitude",
        "total_frp",
    )
    .head(20)
)

acq_date,cluster_id,province,kabupaten_kota,detection_count,centroid_latitude,centroid_longitude,total_frp
date,str,str,str,i64,f64,f64,f64
2026-08-16,"""C02165""","""West Kalimantan""","""Ketapang""",325,-1.758317,110.127543,2836.57
2026-08-18,"""C03897""","""Central Kalimantan""","""Kota Palangka Raya""",319,-2.238938,113.868661,2212.33
2026-08-18,"""C02165""","""West Kalimantan""","""Ketapang""",301,-1.70503,110.159893,2423.92
2026-08-24,"""C03611""","""West Kalimantan""","""Ketapang""",291,-1.833427,110.179151,3531.22
2026-08-24,"""C05965""","""South Sumatra""","""Ogan Komering Ilir""",193,-3.599573,105.722662,3153.95
…,…,…,…,…,…,…,…
2026-08-17,"""C03006""","""Central Kalimantan""","""Kotawaringin Timur""",165,-2.920754,112.883101,1561.48
2026-08-18,"""C03006""","""Central Kalimantan""","""Kotawaringin Timur""",164,-2.91767,112.877394,1626.24
2026-08-20,"""C02165""","""West Kalimantan""","""Ketapang""",162,-1.735279,110.16733,1609.85


In [66]:
trackz = clusterz.unique(
    subset=["cluster_id"]
)

display(
    trackz
    .group_by("active_days")
    .len()
    .sort("active_days")
)

display(
    trackz.select(
        pl.col("active_days").max().alias("max"),
        pl.col("active_days").mean().alias("mean"),
        pl.col("active_days").median().alias("median"),
    )
)

active_days,len
u32,u32
1,4935
2,1378
3,570
4,293
5,166
…,…
25,2
26,2
27,2


max,mean,median
u32,f64,f64
29,1.910093,1.0


In [64]:
top_cluster = (
    trackz
    .sort("track_detection_count", descending=True)
    .get_column("cluster_id")
    .first()
)

(
    clusterz
    .filter(pl.col("cluster_id") == top_cluster)
    .select(
        "acq_date",
        "province",
        "kabupaten_kota",
        "detection_count",
        "centroid_latitude",
        "centroid_longitude",
        "total_frp",
        "active_days",
    )
    .sort("acq_date")
)

acq_date,province,kabupaten_kota,detection_count,centroid_latitude,centroid_longitude,total_frp,active_days
date,str,str,i64,f64,f64,f64,u32
2026-08-09,"""Central Kalimantan""","""Kotawaringin Timur""",2,-2.95272,112.918615,5.77,16
2026-08-10,"""Central Kalimantan""","""Kotawaringin Timur""",16,-2.983132,112.880875,66.61,16
2026-08-11,"""Central Kalimantan""","""Kotawaringin Timur""",19,-2.976134,112.891159,135.0,16
2026-08-12,"""Central Kalimantan""","""Kotawaringin Timur""",68,-3.026468,112.87703,790.68,16
2026-08-13,"""Central Kalimantan""","""Kotawaringin Timur""",39,-2.972077,112.868896,292.91,16
…,…,…,…,…,…,…,…
2026-08-20,"""Central Kalimantan""","""Kotawaringin Timur""",180,-2.907487,112.885783,1664.43,16
2026-08-21,"""Central Kalimantan""","""Kotawaringin Timur""",124,-2.896027,112.890812,1259.78,16
2026-08-22,"""Central Kalimantan""","""Kotawaringin Timur""",95,-2.888691,112.89777,768.16,16


In [65]:
clustered = clusterz.get_column(
    "detection_count"
).sum()

total = firms30d.height

print(f"Total detections:     {total:,}")
print(f"Clustered detections: {clustered:,}")
print(f"Noise detections:     {total - clustered:,}")
print(f"Clustered:            {clustered / total:.1%}")

Total detections:     101,125
Clustered detections: 85,007
Noise detections:     16,118
Clustered:            84.1%


In [72]:
from karhutlawatch.monitoring import (
    cluster_day,
    haversine_km,
)

results = []

for current_date in (
    firms30d
    .get_column("acq_date")
    .unique()
    .sort()
    .to_list()
):
    daily = firms30d.filter(
        pl.col("acq_date") == current_date
    )

    clustered = (
        cluster_day(daily)
        .filter(pl.col("daily_cluster") >= 0)
    )

    bounds = (
        clustered
        .group_by("daily_cluster")
        .agg(
            pl.len().alias("detections"),
            pl.col("latitude").min().alias("lat_min"),
            pl.col("latitude").max().alias("lat_max"),
            pl.col("longitude").min().alias("lon_min"),
            pl.col("longitude").max().alias("lon_max"),
            pl.col("province").mode().first().alias("province"),
            pl.col("kabupaten_kota").mode().first().alias("kabupaten_kota"),
        )
    )

    for row in bounds.iter_rows(named=True):
        row["acq_date"] = current_date
        row["approx_span_km"] = haversine_km(
            row["lat_min"],
            row["lon_min"],
            row["lat_max"],
            row["lon_max"],
        )
        results.append(row)

spans = pl.DataFrame(results)

In [73]:
display(
    spans.select(
        pl.col("approx_span_km").median().alias("median"),
        pl.col("approx_span_km").quantile(0.90).alias("p90"),
        pl.col("approx_span_km").quantile(0.95).alias("p95"),
        pl.col("approx_span_km").quantile(0.99).alias("p99"),
        pl.col("approx_span_km").max().alias("max"),
    )
)

display(
    (
    spans
    .sort("approx_span_km", descending=True)
    .select(
        "acq_date",
        "province",
        "kabupaten_kota",
        "detections",
        "approx_span_km",
    )
    .head(20)
)
)

median,p90,p95,p99,max
f64,f64,f64,f64,f64
1.392303,7.443542,10.927243,20.553681,64.369301


acq_date,province,kabupaten_kota,detections,approx_span_km
str,str,str,i64,f64
"""2026-08-05""","""West Kalimantan""","""Sanggau""",144,64.369301
"""2026-08-06""","""West Kalimantan""","""Landak""",118,62.544054
"""2026-08-22""","""West Kalimantan""","""Ketapang""",179,61.661725
"""2026-08-02""","""West Kalimantan""","""Sanggau""",71,57.964282
"""2026-08-16""","""West Kalimantan""","""Ketapang""",325,54.168462
…,…,…,…,…
"""2026-08-18""","""West Kalimantan""","""Ketapang""",187,41.538019
"""2026-08-16""","""Papua""","""Merauke""",41,40.889973
"""2026-08-18""","""West Kalimantan""","""Ketapang""",301,40.848937


In [74]:
monitoring = pl.read_parquet(
    "../data/analytics/monitoring_areas.parquet"
)

province,kabupaten_kota,recent_detection_count,previous_detection_count,detection_change_pct,recent_active_days,persistent_cluster_count,max_cluster_active_days,recent_total_frp,hours_since_last_detection,monitoring_priority
str,str,u32,u32,f64,u32,u32,u32,f64,i64,f64
"""West Kalimantan""","""Ketapang""",5051,2681,88.399851,7,91,24,45596.22,1,96.5
"""Central Kalimantan""","""Kotawaringin Timur""",2009,1070,87.757009,7,25,17,13771.71,1,95.6
"""Central Kalimantan""","""Kapuas""",1628,565,188.141593,7,37,17,12181.07,1,95.5
"""East Kalimantan""","""Berau""",1187,687,72.780204,7,24,13,11555.62,1,94.3
"""Central Kalimantan""","""Katingan""",622,300,107.333333,7,20,16,6659.04,1,94.1
…,…,…,…,…,…,…,…,…,…,…
"""East Nusa Tenggara""","""Sumba Timur""",456,177,157.627119,7,25,10,3301.72,1,91.2
"""Papua""","""Nabire""",597,43,1288.372093,7,17,7,3611.65,3,91.0
"""South Sumatra""","""Muara Enim""",376,136,176.470588,7,14,19,2180.62,1,90.9


In [75]:
display(
    monitoring.select(
        "province",
        "kabupaten_kota",
        "recent_detection_count",
        "previous_detection_count",
        "detection_change_pct",
        "recent_active_days",
        "persistent_cluster_count",
        "max_cluster_active_days",
        "recent_total_frp",
        "hours_since_last_detection",
        "monitoring_priority",
    ).head(20)
)

display(
    monitoring.select(
        pl.col("monitoring_priority").min().alias("min"),
        pl.col("monitoring_priority").median().alias("median"),
        pl.col("monitoring_priority").mean().alias("mean"),
        pl.col("monitoring_priority").max().alias("max"),
    )
)

display(
    monitoring
    .sort("monitoring_priority", descending=True)
    .select(
        "province",
        "kabupaten_kota",
        "recent_detection_count",
        "detection_change_pct",
        "persistent_cluster_count",
        "max_cluster_active_days",
        "recent_total_frp",
        "monitoring_priority",
    )
    .head(20)
)

display(
    monitoring
    .filter(pl.col("is_new_activity"))
    .sort("recent_detection_count", descending=True)
    .select(
        "province",
        "kabupaten_kota",
        "recent_detection_count",
        "recent_active_days",
        "recent_total_frp",
        "monitoring_priority",
    )
    .head(20)
)

province,kabupaten_kota,recent_detection_count,previous_detection_count,detection_change_pct,recent_active_days,persistent_cluster_count,max_cluster_active_days,recent_total_frp,hours_since_last_detection,monitoring_priority
str,str,u32,u32,f64,u32,u32,u32,f64,i64,f64
"""West Kalimantan""","""Ketapang""",5051,2681,88.399851,7,91,24,45596.22,1,96.5
"""Central Kalimantan""","""Kotawaringin Timur""",2009,1070,87.757009,7,25,17,13771.71,1,95.6
"""Central Kalimantan""","""Kapuas""",1628,565,188.141593,7,37,17,12181.07,1,95.5
"""East Kalimantan""","""Berau""",1187,687,72.780204,7,24,13,11555.62,1,94.3
"""Central Kalimantan""","""Katingan""",622,300,107.333333,7,20,16,6659.04,1,94.1
…,…,…,…,…,…,…,…,…,…,…
"""East Nusa Tenggara""","""Sumba Timur""",456,177,157.627119,7,25,10,3301.72,1,91.2
"""Papua""","""Nabire""",597,43,1288.372093,7,17,7,3611.65,3,91.0
"""South Sumatra""","""Muara Enim""",376,136,176.470588,7,14,19,2180.62,1,90.9


min,median,mean,max
f64,f64,f64,f64
6.8,49.7,50.115242,96.5


province,kabupaten_kota,recent_detection_count,detection_change_pct,persistent_cluster_count,max_cluster_active_days,recent_total_frp,monitoring_priority
str,str,u32,f64,u32,u32,f64,f64
"""West Kalimantan""","""Ketapang""",5051,88.399851,91,24,45596.22,96.5
"""Central Kalimantan""","""Kotawaringin Timur""",2009,87.757009,25,17,13771.71,95.6
"""Central Kalimantan""","""Kapuas""",1628,188.141593,37,17,12181.07,95.5
"""East Kalimantan""","""Berau""",1187,72.780204,24,13,11555.62,94.3
"""Central Kalimantan""","""Katingan""",622,107.333333,20,16,6659.04,94.1
…,…,…,…,…,…,…,…
"""East Nusa Tenggara""","""Sumba Timur""",456,157.627119,25,10,3301.72,91.2
"""Papua""","""Nabire""",597,1288.372093,17,7,3611.65,91.0
"""South Sumatra""","""Muara Enim""",376,176.470588,14,19,2180.62,90.9


province,kabupaten_kota,recent_detection_count,recent_active_days,recent_total_frp,monitoring_priority
str,str,u32,u32,f64,f64
"""Aceh""","""Pidie""",25,1,262.31,57.2
"""West Kalimantan""","""Kota Singkawang""",19,4,30.92,54.3
"""Aceh""","""Aceh Selatan""",13,2,81.33,47.1
"""Papua""","""Waropen""",12,4,41.38,48.3
"""Aceh""","""Aceh Jaya""",11,4,77.5,45.0
…,…,…,…,…,…
"""East Nusa Tenggara""","""Manggarai""",4,4,18.26,23.9
"""Bangka-Belitung Islands""","""Kota Pangkal Pinang""",3,1,49.64,32.6
"""Riau""","""Kota Pekanbaru""",3,1,11.9,26.8


In [8]:
latest_date = firms30d.select(
    pl.col("acq_date").max()
).item()

firms30d.filter(
    pl.col("acq_date") == latest_date
).select(
    pl.len().alias("detections"),
    pl.col("acq_time").min().alias("earliest_time"),
    pl.col("acq_time").max().alias("latest_time"),
    pl.col("acquired_at_utc").max().alias("latest_timestamp"),
)

detections,earliest_time,latest_time,latest_timestamp
u32,i64,i64,"datetime[μs, UTC]"
1868,332,516,2026-08-25 05:16:00 UTC


In [15]:
firms30d = pl.read_parquet("../data/analytics/firms_30d.parquet")
firms30d.columns

['latitude',
 'longitude',
 'bright_ti4',
 'scan',
 'track',
 'acq_date',
 'acq_time',
 'satellite',
 'instrument',
 'confidence',
 'version',
 'bright_ti5',
 'frp',
 'daynight',
 'acquired_at_utc',
 'geometry',
 'province',
 'kabupaten_kota']

In [16]:
(
    firms30d
    .select(pl.col('acq_time'))
    .unique(pl.col('acq_time'))
    .sort(pl.col('acq_time'), descending=True)
)

acq_time
i64
1953
1936
1932
1930
1925
…
332
325
323


In [17]:
times = (
    firms30d
    .select(
        "acq_date",
        "acquired_at_utc",
        "longitude",
        "latitude",
    )
    .unique(["acq_date", "acquired_at_utc"])
    .sort("acquired_at_utc")
    .with_columns(
        (
            pl.col("acquired_at_utc")
            - pl.col("acquired_at_utc").shift(1)
        )
        .dt.total_minutes()
        .alias("gap_minutes")
    )
    .with_columns(
        (
            pl.col("gap_minutes").is_null()
            | (pl.col("gap_minutes") > 30)
            | (
                pl.col("acq_date")
                != pl.col("acq_date").shift(1)
            )
        )
        .cast(pl.Int32)
        .cum_sum()
        .alias("pass_id")
    )
)

passes = (
    times
    .group_by("pass_id")
    .agg(
        pl.col("acq_date").first(),
        pl.col("acquired_at_utc").min().alias("pass_start"),
        pl.col("acquired_at_utc").max().alias("pass_end"),
        pl.col("longitude").min().alias("min_lon"),
        pl.col("longitude").max().alias("max_lon"),
        pl.len().alias("timestamps"),
    )
    .sort("pass_start")
)

passes

pass_id,acq_date,pass_start,pass_end,min_lon,max_lon,timestamps
i32,str,"datetime[μs, UTC]","datetime[μs, UTC]",f64,f64,u32
1,"""2026-07-28""",2026-07-28 03:55:00 UTC,2026-07-28 03:57:00 UTC,136.37315,140.79227,2
2,"""2026-07-28""",2026-07-28 05:36:00 UTC,2026-07-28 05:40:00 UTC,106.61295,120.15936,3
3,"""2026-07-28""",2026-07-28 07:18:00 UTC,2026-07-28 07:22:00 UTC,96.46567,107.30562,3
4,"""2026-07-28""",2026-07-28 16:37:00 UTC,2026-07-28 16:41:00 UTC,123.88424,140.31961,3
5,"""2026-07-28""",2026-07-28 18:19:00 UTC,2026-07-28 18:21:00 UTC,112.01978,115.10916,2
…,…,…,…,…,…,…
161,"""2026-08-26""",2026-08-26 04:51:00 UTC,2026-08-26 04:55:00 UTC,120.08438,123.33575,3
162,"""2026-08-26""",2026-08-26 06:33:00 UTC,2026-08-26 06:40:00 UTC,95.60567,119.63406,4
163,"""2026-08-26""",2026-08-26 15:56:00 UTC,2026-08-26 15:58:00 UTC,140.36172,140.86894,2


In [18]:
(
    passes
    .with_columns(
        pl.col("pass_start").dt.hour().alias("hour_utc")
    )
    .group_by("hour_utc")
    .agg(
        pl.len().alias("passes"),
        pl.col("acq_date").n_unique().alias("days_seen"),
    )
    .sort("hour_utc")
)

hour_utc,passes,days_seen
i8,u32,u32
3,14,14
4,18,18
5,17,17
6,19,19
7,14,14
15,16,16
16,19,19
17,17,17
18,18,18


In [19]:
(
    passes
    .group_by("acq_date")
    .agg(
        pl.col("pass_start").min().alias("first_pass"),
        pl.col("pass_end").max().alias("last_pass"),
        pl.len().alias("passes"),
    )
    .sort("acq_date")
)

acq_date,first_pass,last_pass,passes
str,"datetime[μs, UTC]","datetime[μs, UTC]",u32
"""2026-07-28""",2026-07-28 03:55:00 UTC,2026-07-28 18:21:00 UTC,5
"""2026-07-29""",2026-07-29 03:36:00 UTC,2026-07-29 18:04:00 UTC,5
"""2026-07-30""",2026-07-30 03:17:00 UTC,2026-07-30 19:23:00 UTC,6
"""2026-07-31""",2026-07-31 04:40:00 UTC,2026-07-31 19:06:00 UTC,5
"""2026-08-01""",2026-08-01 04:19:00 UTC,2026-08-01 18:47:00 UTC,6
…,…,…,…
"""2026-08-22""",2026-08-22 04:27:00 UTC,2026-08-22 18:53:00 UTC,6
"""2026-08-23""",2026-08-23 04:08:00 UTC,2026-08-23 18:36:00 UTC,6
"""2026-08-24""",2026-08-24 03:49:00 UTC,2026-08-24 19:53:00 UTC,6


In [20]:
firms_with_pass = firms30d.join(
    times.select(
        "acquired_at_utc",
        "pass_id",
    ).unique(),
    on="acquired_at_utc",
    how="left",
)

In [21]:
selected_date = "2026-08-26"

day_passes = (
    firms_with_pass
    .filter(pl.col("acq_date") == selected_date)
    .group_by("pass_id")
    .agg(
        pl.col("acquired_at_utc").min().alias("start"),
        pl.col("acquired_at_utc").max().alias("end"),

        pl.col("longitude").min().alias("min_lon"),
        pl.col("longitude").max().alias("max_lon"),

        pl.col("latitude").min().alias("min_lat"),
        pl.col("latitude").max().alias("max_lat"),

        pl.col("province").n_unique().alias("province_count"),
        pl.col("province").unique().sort().alias("provinces"),

        pl.len().alias("detections"),
    )
    .sort("start")
)

day_passes

pass_id,start,end,min_lon,max_lon,min_lat,max_lat,province_count,provinces,detections
i32,"datetime[μs, UTC]","datetime[μs, UTC]",f64,f64,f64,f64,u32,list[str],u32
161,2026-08-26 04:51:00 UTC,2026-08-26 04:55:00 UTC,115.26678,140.98097,-10.62769,2.08033,15,"[""Central Kalimantan"", ""Central Sulawesi"", … ""West Sulawesi""]",844
162,2026-08-26 06:33:00 UTC,2026-08-26 06:40:00 UTC,95.37653,119.63406,-9.5121,5.5969,18,"[""Aceh"", ""Bangka-Belitung Islands"", … ""West Nusa Tenggara""]",2466
163,2026-08-26 15:56:00 UTC,2026-08-26 15:58:00 UTC,130.15009,140.86894,-8.38796,-0.56249,3,"[""Maluku"", ""Papua"", ""West Papua""]",172
164,2026-08-26 17:34:00 UTC,2026-08-26 17:39:00 UTC,103.61459,130.50523,-10.14225,2.78093,24,"[""Bangka-Belitung Islands"", ""Banten"", … ""West Sulawesi""]",1961
165,2026-08-26 19:17:00 UTC,2026-08-26 19:19:00 UTC,96.90461,105.91281,-4.92299,5.15266,6,"[""Aceh"", ""Bangka-Belitung Islands"", … ""South Sumatra""]",171


In [25]:
import altair as alt

alt.data_transformers.disable_max_rows()
selected_date = "2026-08-26"

plot_df = (
    firms_with_pass
    .filter(pl.col("acq_date") == selected_date)
    .select(
        "longitude",
        "latitude",
        "pass_id",
        "acquired_at_utc",
        "province",
    )
    .to_pandas()
)

alt.Chart(plot_df).mark_circle(
    size=20,
    opacity=0.5,
).encode(
    x=alt.X(
        "longitude:Q",
        title="Longitude",
        scale=alt.Scale(domain=[94, 142]),
    ),
    y=alt.Y(
        "latitude:Q",
        title="Latitude",
        scale=alt.Scale(domain=[-12, 7]),
    ),
    color=alt.Color(
        "pass_id:N",
        title="Pass",
    ),
    tooltip=[
        "pass_id:N",
        "acquired_at_utc:T",
        "province:N",
        "longitude:Q",
        "latitude:Q",
    ],
).properties(
    width=900,
    height=400,
)

alt.Chart(...)